In [1]:
from IPython.display import display

from src.phase3.experiment import (
    PHASE3_TEST_DATA_DIR,
    PHASE3_TEST_RESULTS_ROOT,
    build_phase3_all_experiment_specs,
    build_phase3_combined_experiment_specs,
    build_phase3_individual_experiment_specs,
    prepare_phase3_experiment_dataset,
    run_phase3_experiment,
    select_best_phase3_individual_options,
    select_best_phase3_spec,
    summarize_phase3_experiment_specs,
)

FORCE_REBUILD_DATASETS = False
FORCE_SCORE = False
FORCE_TRAIN = False

print(f"CSV output dir: data/{PHASE3_TEST_DATA_DIR}")
print(f"Training output root: results/<architecture>/{PHASE3_TEST_RESULTS_ROOT}")

CSV output dir: data/phase3test
Training output root: results/<architecture>/phase3_test


## 1. Definir experimentos individuales

Se prueban por separado `top%`, `dedup` y `quality` sobre cada dataset de partida de fase 2.

In [2]:
individual_specs = build_phase3_individual_experiment_specs()

print(f"Individual datasets: {len(individual_specs)}")
for spec in individual_specs:
    print(f"- {spec.descriptor}: {spec.output_csv}")

Individual datasets: 18
- kinf_top60: phase3test\phase3_kinf_top60.csv
- kinf_top70: phase3test\phase3_kinf_top70.csv
- kinf_top80: phase3test\phase3_kinf_top80.csv
- kinf_dedup_config: phase3test\phase3_kinf_dedup_config.csv
- kinf_dedup_p90_10: phase3test\phase3_kinf_dedup_p90_10.csv
- kinf_dedup_p75_25: phase3test\phase3_kinf_dedup_p75_25.csv
- kinf_quality_config: phase3test\phase3_kinf_quality_config.csv
- kinf_quality_p10: phase3test\phase3_kinf_quality_p10.csv
- kinf_quality_p25: phase3test\phase3_kinf_quality_p25.csv
- conf060_top60: phase3test\phase3_conf060_top60.csv
- conf060_top70: phase3test\phase3_conf060_top70.csv
- conf060_top80: phase3test\phase3_conf060_top80.csv
- conf060_dedup_config: phase3test\phase3_conf060_dedup_config.csv
- conf060_dedup_p90_10: phase3test\phase3_conf060_dedup_p90_10.csv
- conf060_dedup_p75_25: phase3test\phase3_conf060_dedup_p75_25.csv
- conf060_quality_config: phase3test\phase3_conf060_quality_config.csv
- conf060_quality_p10: phase3test\phas

## 2. Crear CSVs individuales

Los CSVs se guardan en `data/phase3test`. Si ya existen y `FORCE_REBUILD_DATASETS=False`, se reutilizan.

In [3]:
prepared_individual = []

for index, spec in enumerate(individual_specs, start=1):
    print(f"[{index}/{len(individual_specs)}] Preparing {spec.descriptor}")
    result = prepare_phase3_experiment_dataset(
        spec,
        force_rebuild=FORCE_REBUILD_DATASETS,
        force_score=FORCE_SCORE,
    )
    prepared_individual.append(result)
    print(f"    output_csv={result['output_csv']} | reused={result['reused']}")

[1/18] Preparing kinf_top60
    output_csv=phase3test\phase3_kinf_top60.csv | reused=True
[2/18] Preparing kinf_top70
    output_csv=phase3test\phase3_kinf_top70.csv | reused=True
[3/18] Preparing kinf_top80
    output_csv=phase3test\phase3_kinf_top80.csv | reused=True
[4/18] Preparing kinf_dedup_config
    output_csv=phase3test\phase3_kinf_dedup_config.csv | reused=True
[5/18] Preparing kinf_dedup_p90_10
    output_csv=phase3test\phase3_kinf_dedup_p90_10.csv | reused=True
[6/18] Preparing kinf_dedup_p75_25
    output_csv=phase3test\phase3_kinf_dedup_p75_25.csv | reused=True
[7/18] Preparing kinf_quality_config
    output_csv=phase3test\phase3_kinf_quality_config.csv | reused=True
[8/18] Preparing kinf_quality_p10
    output_csv=phase3test\phase3_kinf_quality_p10.csv | reused=True
[9/18] Preparing kinf_quality_p25
    output_csv=phase3test\phase3_kinf_quality_p25.csv | reused=True
[10/18] Preparing conf060_top60
    output_csv=phase3test\phase3_conf060_top60.csv | reused=True
[11/18] P

## 3. Entrenar individuales y mostrar metricas

Cada experimento usa 2 seeds. Los plots se guardan por el entrenamiento, pero no se muestran.

In [4]:
individual_metric_tables = {}

for index, spec in enumerate(individual_specs, start=1):
    print("=" * 100)
    print(f"[{index}/{len(individual_specs)}] Training {spec.descriptor}")
    metrics_df = run_phase3_experiment(
        spec,
        force_rebuild_dataset=False,
        force_score=FORCE_SCORE,
        force_train=FORCE_TRAIN,
    )
    individual_metric_tables[spec.descriptor] = metrics_df
    display(metrics_df)

[1/18] Training kinf_top60
Rows: 4889
histology
Adenoma                     2107
Sessile_serrated_adenoma    1446
Hyperplastic                 861
Adenocarcinoma               475
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top60\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top60\seed_2\best_baseline_model.pth


,scope,accuracy,mcc,macro_f1,precision,recall
0,general,0.5471 +/- 0.0083,0.2704 +/- 0.0122,0.5716 +/- 0.0100,0.5744 +/- 0.0140,0.5795 +/- 0.0066
1,Adenoma,0.6331 +/- 0.0031,0.2807 +/- 0.0090,0.6503 +/- 0.0005,0.7285 +/- 0.0069,0.5873 +/- 0.0036
2,Sessile_serrated_adenoma,0.6868 +/- 0.0104,0.2626 +/- 0.0075,0.4788 +/- 0.0133,0.4322 +/- 0.0084,0.5385 +/- 0.0466
3,Hyperplastic,0.7794 +/- 0.0229,0.1277 +/- 0.0183,0.2504 +/- 0.0089,0.2173 +/- 0.0222,0.2976 +/- 0.0168
4,Adenocarcinoma,0.9949 +/- 0.0010,0.9044 +/- 0.0179,0.9068 +/- 0.0171,0.9196 +/- 0.0351,0.8947 +/- 0.0000


[2/18] Training kinf_top70
Rows: 5245
histology
Adenoma                     2195
Sessile_serrated_adenoma    1566
Hyperplastic                 947
Adenocarcinoma               537
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top70\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top70\seed_2\best_baseline_model.pth


,scope,accuracy,mcc,macro_f1,precision,recall
0,general,0.5375 +/- 0.0031,0.2593 +/- 0.0167,0.5707 +/- 0.0122,0.5751 +/- 0.0267,0.5787 +/- 0.0034
1,Adenoma,0.6294 +/- 0.0083,0.2763 +/- 0.0247,0.6441 +/- 0.0009,0.7289 +/- 0.0194,0.5772 +/- 0.0107
2,Sessile_serrated_adenoma,0.6779 +/- 0.0021,0.2426 +/- 0.0127,0.4658 +/- 0.0108,0.4188 +/- 0.0051,0.5247 +/- 0.0194
3,Hyperplastic,0.7713 +/- 0.0052,0.1122 +/- 0.0222,0.2390 +/- 0.0222,0.2028 +/- 0.0115,0.2917 +/- 0.0421
4,Adenocarcinoma,0.9963 +/- 0.0010,0.9328 +/- 0.0165,0.9338 +/- 0.0151,0.9500 +/- 0.0707,0.9211 +/- 0.0372


[3/18] Training kinf_top80
Rows: 5603
histology
Adenoma                     2283
Sessile_serrated_adenoma    1686
Hyperplastic                1035
Adenocarcinoma               599
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top80\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top80\seed_2\best_baseline_model.pth


...

,scope,accuracy,mcc,macro_f1,precision,recall
0,general,0.5647 +/- 0.0042,0.2811 +/- 0.0094,0.5793 +/- 0.0006,0.5802 +/- 0.0036,0.5843 +/- 0.0085
1,Adenoma,0.6360 +/- 0.0010,0.2762 +/- 0.0127,0.6629 +/- 0.0091,0.7177 +/- 0.0151,0.6165 +/- 0.0269
2,Sessile_serrated_adenoma,0.6971 +/- 0.0062,0.2845 +/- 0.0037,0.4937 +/- 0.0072,0.4469 +/- 0.0061,0.5522 +/- 0.0272
3,Hyperplastic,0.8015 +/- 0.0021,0.1406 +/- 0.0207,0.2537 +/- 0.0213,0.2367 +/- 0.0119,0.2738 +/- 0.0337
4,Adenocarcinoma,0.9949 +/- 0.0010,0.9044 +/- 0.0179,0.9068 +/- 0.0171,0.9196 +/- 0.0351,0.8947 +/- 0.0000


[17/18] Training conf060_quality_p10
Rows: 10076
histology
Adenoma                     6446
Sessile_serrated_adenoma    2302
Hyperplastic                1040
Adenocarcinoma               288
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\conf060_quality_p10\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\conf060_quality_p10\seed_2\best_baseline_model.pth


,scope,accuracy,mcc,macro_f1,precision,recall
0,general,0.5346 +/- 0.0177,0.2437 +/- 0.0192,0.5523 +/- 0.0104,0.5435 +/- 0.0141,0.5705 +/- 0.0047
1,Adenoma,0.6015 +/- 0.0187,0.2146 +/- 0.0329,0.6219 +/- 0.0230,0.6923 +/- 0.0139,0.5646 +/- 0.0286
2,Sessile_serrated_adenoma,0.6772 +/- 0.0031,0.2566 +/- 0.0122,0.4792 +/- 0.0094,0.4217 +/- 0.0056,0.5549 +/- 0.0155
3,Hyperplastic,0.7985 +/- 0.0125,0.1325 +/- 0.0002,0.2470 +/- 0.0060,0.2301 +/- 0.0082,0.2679 +/- 0.0253
4,Adenocarcinoma,0.9919 +/- 0.0010,0.8575 +/- 0.0154,0.8609 +/- 0.0154,0.8298 +/- 0.0286,0.8947 +/- 0.0000


[18/18] Training conf060_quality_p25
Rows: 6045
histology
Adenoma                     3989
Sessile_serrated_adenoma    1318
Hyperplastic                 595
Adenocarcinoma               143
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\conf060_quality_p25\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\conf060_quality_p25\seed_2\best_baseline_model.pth


,scope,accuracy,mcc,macro_f1,precision,recall
0,general,0.5346 +/- 0.0135,0.2522 +/- 0.0078,0.5005 +/- 0.0061,0.5174 +/- 0.0094,0.4994 +/- 0.0002
1,Adenoma,0.6169 +/- 0.0073,0.2523 +/- 0.0059,0.6306 +/- 0.0155,0.7167 +/- 0.0034,0.5633 +/- 0.0269
2,Sessile_serrated_adenoma,0.6890 +/- 0.0094,0.2925 +/- 0.0172,0.5053 +/- 0.0108,0.4400 +/- 0.0121,0.5934 +/- 0.0078
3,Hyperplastic,0.7816 +/- 0.0094,0.1048 +/- 0.0117,0.2281 +/- 0.0151,0.2026 +/- 0.0036,0.2619 +/- 0.0337
4,Adenocarcinoma,0.9816 +/- 0.0010,0.6320 +/- 0.0153,0.6378 +/- 0.0131,0.7104 +/- 0.0324,0.5789 +/- 0.0000


## 4. Resumen individual y seleccion de mejores parametros

La seleccion usa `macro_f1`; si la diferencia es menor que `0.005`, desempata por `mcc`.

In [5]:
individual_summary = summarize_phase3_experiment_specs(individual_specs)
display(individual_summary)

best_options = select_best_phase3_individual_options(individual_summary)
print("Best top fraction:", best_options["best_top_fraction"])
print("Best dedup mode:", best_options["best_dedup_mode"])
print("Best quality mode:", best_options["best_quality_mode"])

display(best_options["best_top_row"].to_frame().T)
display(best_options["best_dedup_row"].to_frame().T)
display(best_options["best_quality_row"].to_frame().T)

,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,accuracy,accuracy_std,mcc,mcc_std,macro_f1,macro_f1_std,precision,precision_std,recall,recall_std
0,conf060_dedup_config,conf060,dedup,NaN,config,NaN,phase3test\phase3_conf060_dedup_config.csv,0.547059,0.008319,0.282075,0.018841,0.584553,0.018398,0.596198,0.018707,0.589808,0.020636
1,conf060_dedup_p75_25,conf060,dedup,NaN,p75_25,NaN,phase3test\phase3_conf060_dedup_p75_25.csv,0.561765,0.004159,0.287859,0.000218,0.584284,0.006934,0.585882,0.011341,0.592226,0.005276
2,kinf_dedup_p75_25,kinf,dedup,NaN,p75_25,NaN,phase3test\phase3_kinf_dedup_p75_25.csv,0.556618,0.003120,0.276157,0.013585,0.579952,0.005779,0.584390,0.006643,0.583835,0.010780
3,conf060_quality_config,conf060,quality,NaN,NaN,config,phase3test\phase3_conf060_quality_config.csv,0.564706,0.004159,0.281073,0.009385,0.579297,0.000578,0.580212,0.003558,0.584300,0.008504
4,conf060_top80,conf060,top,0.8,NaN,NaN,phase3test\phase3_conf060_top80.csv,0.536029,0.009359,0.268520,0.020073,0.574968,0.001884,0.566525,0.000462,0.600145,0.006475
5,conf060_dedup_p90_10,conf060,dedup,NaN,p90_10,NaN,phase3test\phase3_conf060_dedup_p90_10.csv,0.551471,0.008319,0.273570,0.005332,0.574805,0.003443,0.576944,0.009106,0.582799,0.003924
6,kinf_top60,kinf,top,0.6,NaN,NaN,phase3test\phase3_kinf_top60.csv,0.547059,0.008319,0.270439,0.012243,0.571595,0.009967,0.574375,0.013971,0.579540,0.006552
7,kinf_top70,kinf,top,0.7,NaN,NaN,phase3test\phase3_kinf_top70.csv,0.537500,0.003120,0.259341,0.016655,0.570662,0.012247,0.575121,0.026673,0.578665,0.003390
8,kinf_dedup_config,kinf,dedup,NaN,config,NaN,phase3test\phase3_kinf_dedup_config.csv,0.533824,0.004159,0.268355,0.004770,0.570112,0.007917,0.596153,0.001498,0.567392,0.011523
9,kinf_dedup_p90_10,kinf,dedup,NaN,p90_10,NaN,phase3test\phase3_kinf_dedup_p90_10.csv,0.549265,0.005199,0.262994,0.024353,0.567676,0.017214,0.574567,0.037181,0.570153,0.004723


Best top fraction: 0.6
Best dedup mode: p75_25
Best quality mode: config


,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,accuracy,accuracy_std,mcc,mcc_std,macro_f1,macro_f1_std,precision,precision_std,recall,recall_std
1,kinf_top60,kinf,top,0.6,NaN,NaN,phase3test\phase3_kinf_top60.csv,0.547059,0.008319,0.270439,0.012243,0.571595,0.009967,0.574375,0.013971,0.57954,0.006552


,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,accuracy,accuracy_std,mcc,mcc_std,macro_f1,macro_f1_std,precision,precision_std,recall,recall_std
1,conf060_dedup_p75_25,conf060,dedup,NaN,p75_25,NaN,phase3test\phase3_conf060_dedup_p75_25.csv,0.561765,0.004159,0.287859,0.000218,0.584284,0.006934,0.585882,0.011341,0.592226,0.005276


,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,accuracy,accuracy_std,mcc,mcc_std,macro_f1,macro_f1_std,precision,precision_std,recall,recall_std
0,conf060_quality_config,conf060,quality,NaN,NaN,config,phase3test\phase3_conf060_quality_config.csv,0.564706,0.004159,0.281073,0.009385,0.579297,0.000578,0.580212,0.003558,0.5843,0.008504


## 5. Definir combinaciones finales

Con los mejores valores encontrados, se prueban solo combinaciones de 2 o 3 condiciones: `top+dedup`, `top+quality`, `dedup+quality`, `top+dedup+quality`.

In [6]:
combined_specs = build_phase3_combined_experiment_specs(
    best_top_fraction=best_options["best_top_fraction"],
    best_dedup_mode=best_options["best_dedup_mode"],
    best_quality_mode=best_options["best_quality_mode"],
)

print(f"Combined datasets: {len(combined_specs)}")
for spec in combined_specs:
    print(f"- {spec.descriptor}: {spec.output_csv}")

Combined datasets: 8
- kinf_top60_dedup_p75_25: phase3test\phase3_kinf_top60_dedup_p75_25.csv
- kinf_top60_quality_config: phase3test\phase3_kinf_top60_quality_config.csv
- kinf_dedup_p75_25_quality_config: phase3test\phase3_kinf_dedup_p75_25_quality_config.csv
- kinf_top60_dedup_p75_25_quality_config: phase3test\phase3_kinf_top60_dedup_p75_25_quality_config.csv
- conf060_top60_dedup_p75_25: phase3test\phase3_conf060_top60_dedup_p75_25.csv
- conf060_top60_quality_config: phase3test\phase3_conf060_top60_quality_config.csv
- conf060_dedup_p75_25_quality_config: phase3test\phase3_conf060_dedup_p75_25_quality_config.csv
- conf060_top60_dedup_p75_25_quality_config: phase3test\phase3_conf060_top60_dedup_p75_25_quality_config.csv


## 6. Crear CSVs combinados

In [7]:
prepared_combined = []

for index, spec in enumerate(combined_specs, start=1):
    print(f"[{index}/{len(combined_specs)}] Preparing {spec.descriptor}")
    result = prepare_phase3_experiment_dataset(
        spec,
        force_rebuild=FORCE_REBUILD_DATASETS,
        force_score=FORCE_SCORE,
    )
    prepared_combined.append(result)
    print(f"    output_csv={result['output_csv']} | reused={result['reused']}")

[1/8] Preparing kinf_top60_dedup_p75_25
    output_csv=phase3test\phase3_kinf_top60_dedup_p75_25.csv | reused=True
[2/8] Preparing kinf_top60_quality_config
    output_csv=phase3test\phase3_kinf_top60_quality_config.csv | reused=True
[3/8] Preparing kinf_dedup_p75_25_quality_config
    output_csv=phase3test\phase3_kinf_dedup_p75_25_quality_config.csv | reused=True
[4/8] Preparing kinf_top60_dedup_p75_25_quality_config
    output_csv=phase3test\phase3_kinf_top60_dedup_p75_25_quality_config.csv | reused=True
[5/8] Preparing conf060_top60_dedup_p75_25
    output_csv=phase3test\phase3_conf060_top60_dedup_p75_25.csv | reused=True
[6/8] Preparing conf060_top60_quality_config
    output_csv=phase3test\phase3_conf060_top60_quality_config.csv | reused=True
[7/8] Preparing conf060_dedup_p75_25_quality_config
    output_csv=phase3test\phase3_conf060_dedup_p75_25_quality_config.csv | reused=True
[8/8] Preparing conf060_top60_dedup_p75_25_quality_config
    output_csv=phase3test\phase3_conf060_top6

## 7. Entrenar combinados y mostrar metricas

In [8]:
combined_metric_tables = {}

for index, spec in enumerate(combined_specs, start=1):
    print("=" * 100)
    print(f"[{index}/{len(combined_specs)}] Training {spec.descriptor}")
    metrics_df = run_phase3_experiment(
        spec,
        force_rebuild_dataset=False,
        force_score=FORCE_SCORE,
        force_train=FORCE_TRAIN,
    )
    combined_metric_tables[spec.descriptor] = metrics_df
    display(metrics_df)

[1/8] Training kinf_top60_dedup_p75_25
Rows: 4638
histology
Adenoma                     2030
Sessile_serrated_adenoma    1383
Hyperplastic                 790
Adenocarcinoma               435
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top60_dedup_p75_25\seed_1\best_baseline_model.pth
Loss weights: {'Adenoma': 0.5568, 'Sessile_serrated_adenoma': 0.7284, 'Hyperplastic': 1.078, 'Adenocarcinoma': 1.6368}
Initiating training phase. Saving results to: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top60_dedup_p75_25\seed_2


Training Progress:  41%|████      | 41/100 [17:55<25:47, 26.23s/epoch, Stage=full_network, Train Loss=0.1513, Val Loss=0.4229, Val Macro F1=0.5507, Selection Score=-0.5507, Best Epoch=26, LR=3.1e-07/3.1e-06]       


Early stopping triggered after 16 epochs without improving macro_f1.
Optimization sequence completed with seed 123. Selected checkpoint macro-F1: 0.5563 with validation loss 0.4281 and validation score -0.5563 at epoch 26.
Artifacts successfully written to 'C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top60_dedup_p75_25\seed_2'.


,scope,accuracy,mcc,macro_f1,precision,recall
0,general,0.5382 +/- 0.0021,0.2549 +/- 0.0121,0.5567 +/- 0.0006,0.5540 +/- 0.0023,0.5691 +/- 0.0067
1,Adenoma,0.6265 +/- 0.0042,0.2676 +/- 0.0153,0.6437 +/- 0.0024,0.7220 +/- 0.0133,0.5810 +/- 0.0125
2,Sessile_serrated_adenoma,0.6757 +/- 0.0031,0.2435 +/- 0.0111,0.4678 +/- 0.0121,0.4172 +/- 0.0002,0.5330 +/- 0.0311
3,Hyperplastic,0.7809 +/- 0.0042,0.1082 +/- 0.0113,0.2320 +/- 0.0090,0.2046 +/- 0.0091,0.2679 +/- 0.0084
4,Adenocarcinoma,0.9934 +/- 0.0010,0.8800 +/- 0.0165,0.8833 +/- 0.0162,0.8724 +/- 0.0316,0.8947 +/- 0.0000


[2/8] Training kinf_top60_quality_config
Rows: 4640
histology
Adenoma                     1968
Sessile_serrated_adenoma    1397
Hyperplastic                 810
Adenocarcinoma               465
Loss weights: {'Adenoma': 0.5816, 'Sessile_serrated_adenoma': 0.7392, 'Hyperplastic': 1.0826, 'Adenocarcinoma': 1.5966}
Initiating training phase. Saving results to: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\kinf_top60_quality_config\seed_1


...

Training Progress:  46%|████▌     | 46/100 [27:29<32:16, 35.86s/epoch, Stage=full_network, Train Loss=0.1045, Val Loss=0.3994, Val Macro F1=0.5229, Selection Score=-0.5229, Best Epoch=31, LR=1.0e-07/7.8e-07]       


Early stopping triggered after 16 epochs without improving macro_f1.
Optimization sequence completed with seed 42. Selected checkpoint macro-F1: 0.5510 with validation loss 0.4019 and validation score -0.5510 at epoch 31.
Artifacts successfully written to 'C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\conf060_top60_dedup_p75_25_quality_config\seed_1'.
Loss weights: {'Adenoma': 0.3736, 'Sessile_serrated_adenoma': 0.619, 'Hyperplastic': 1.0365, 'Adenocarcinoma': 1.9709}
Initiating training phase. Saving results to: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\conf060_top60_dedup_p75_25_quality_config\seed_2


Training Progress:  39%|███▉      | 39/100 [23:33<36:50, 36.23s/epoch, Stage=full_network, Train Loss=0.1005, Val Loss=0.3883, Val Macro F1=0.5149, Selection Score=-0.5149, Best Epoch=24, LR=1.6e-07/1.6e-06]       


Early stopping triggered after 16 epochs without improving macro_f1.
Optimization sequence completed with seed 123. Selected checkpoint macro-F1: 0.5469 with validation loss 0.3830 and validation score -0.5469 at epoch 24.
Artifacts successfully written to 'C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\conf060_top60_dedup_p75_25_quality_config\seed_2'.


,scope,accuracy,mcc,macro_f1,precision,recall
0,general,0.5000 +/- 0.0062,0.2220 +/- 0.0102,0.5490 +/- 0.0029,0.5605 +/- 0.0137,0.5580 +/- 0.0067
1,Adenoma,0.6029 +/- 0.0042,0.2371 +/- 0.0051,0.6041 +/- 0.0074,0.7178 +/- 0.0006,0.5215 +/- 0.0107
2,Sessile_serrated_adenoma,0.6551 +/- 0.0010,0.1944 +/- 0.0268,0.4336 +/- 0.0263,0.3867 +/- 0.0085,0.4945 +/- 0.0544
3,Hyperplastic,0.7463 +/- 0.0198,0.1036 +/- 0.0109,0.2386 +/- 0.0046,0.1901 +/- 0.0118,0.3214 +/- 0.0168
4,Adenocarcinoma,0.9956 +/- 0.0021,0.9181 +/- 0.0373,0.9196 +/- 0.0351,0.9474 +/- 0.0744,0.8947 +/- 0.0000


## 8. Tabla final general

Tabla sin clases, ordenada por `macro_f1` y `mcc`.

In [9]:
all_specs = build_phase3_all_experiment_specs(
    best_top_fraction=best_options["best_top_fraction"],
    best_dedup_mode=best_options["best_dedup_mode"],
    best_quality_mode=best_options["best_quality_mode"],
)

final_summary = summarize_phase3_experiment_specs(all_specs)
display(final_summary)

best_final = select_best_phase3_spec(final_summary)
print("Best final experiment:")
display(best_final.to_frame().T)

,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,accuracy,accuracy_std,mcc,mcc_std,macro_f1,macro_f1_std,precision,precision_std,recall,recall_std
0,conf060_dedup_config,conf060,dedup,NaN,config,NaN,phase3test\phase3_conf060_dedup_config.csv,0.547059,0.008319,0.282075,0.018841,0.584553,0.018398,0.596198,0.018707,0.589808,0.020636
1,conf060_dedup_p75_25,conf060,dedup,NaN,p75_25,NaN,phase3test\phase3_conf060_dedup_p75_25.csv,0.561765,0.004159,0.287859,0.000218,0.584284,0.006934,0.585882,0.011341,0.592226,0.005276
2,kinf_dedup_p75_25,kinf,dedup,NaN,p75_25,NaN,phase3test\phase3_kinf_dedup_p75_25.csv,0.556618,0.003120,0.276157,0.013585,0.579952,0.005779,0.584390,0.006643,0.583835,0.010780
3,conf060_quality_config,conf060,quality,NaN,NaN,config,phase3test\phase3_conf060_quality_config.csv,0.564706,0.004159,0.281073,0.009385,0.579297,0.000578,0.580212,0.003558,0.584300,0.008504
4,conf060_top80,conf060,top,0.8,NaN,NaN,phase3test\phase3_conf060_top80.csv,0.536029,0.009359,0.268520,0.020073,0.574968,0.001884,0.566525,0.000462,0.600145,0.006475
5,conf060_dedup_p90_10,conf060,dedup,NaN,p90_10,NaN,phase3test\phase3_conf060_dedup_p90_10.csv,0.551471,0.008319,0.273570,0.005332,0.574805,0.003443,0.576944,0.009106,0.582799,0.003924
6,conf060_dedup_p75_25_quality_config,conf060,dedup+quality,NaN,p75_25,config,phase3test\phase3_conf060_dedup_p75_25_quality...,0.543382,0.003120,0.265530,0.019951,0.573538,0.012072,0.570316,0.008721,0.588657,0.020836
7,kinf_top60_dedup_p75_25_quality_config,kinf,top+dedup+quality,0.6,p75_25,config,phase3test\phase3_kinf_top60_dedup_p75_25_qual...,0.533824,0.010399,0.259095,0.016168,0.572103,0.016458,0.587151,0.005870,0.572611,0.024807
8,kinf_top60,kinf,top,0.6,NaN,NaN,phase3test\phase3_kinf_top60.csv,0.547059,0.008319,0.270439,0.012243,0.571595,0.009967,0.574375,0.013971,0.579540,0.006552
9,kinf_dedup_p75_25_quality_config,kinf,dedup+quality,NaN,p75_25,config,phase3test\phase3_kinf_dedup_p75_25_quality_co...,0.534559,0.009359,0.257643,0.004173,0.570801,0.003305,0.588419,0.002279,0.569069,0.003381


Best final experiment:


,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,accuracy,accuracy_std,mcc,mcc_std,macro_f1,macro_f1_std,precision,precision_std,recall,recall_std
1,conf060_dedup_p75_25,conf060,dedup,NaN,p75_25,NaN,phase3test\phase3_conf060_dedup_p75_25.csv,0.561765,0.004159,0.287859,0.000218,0.584284,0.006934,0.585882,0.011341,0.592226,0.005276
